# 01 — Veri Temizleme
Her hücreyi sırayla çalıştır.

In [1]:
import pandas as pd
import numpy as np

# Veriyi yükle
df = pd.read_csv('data.csv', encoding='unicode_escape')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f'Ham veri: {len(df):,} satır, {df.shape[1]} sütun')
df.head()

Ham veri: 541,909 satır, 8 sütun


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
# ADIM 1 — Eksik CustomerID
# RFM analizi müşteri bazlıdır, kimliği olmayan satırlar kullanılamaz

print(f'Eksik CustomerID: {df["CustomerID"].isna().sum():,} ({df["CustomerID"].isna().mean()*100:.1f}%)')

df_clean = df.dropna(subset=['CustomerID']).copy()
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

print(f'Çıkarılan : {len(df) - len(df_clean):,} satır')
print(f'Kalan     : {len(df_clean):,} satır')

Eksik CustomerID: 135,080 (24.9%)
Çıkarılan : 135,080 satır
Kalan     : 406,829 satır


In [9]:
# ADIM 2 — Negatif ve sıfır Quantity
# Negatif değerler iade işlemlerini temsil eder
# Bunları ana analizden çıkarıp ayrı bir dosyada saklarız

mask_neg   = df_clean['Quantity'] <= 0
df_returns = df_clean[mask_neg].copy()   # iade veri seti
df_clean   = df_clean[~mask_neg].copy()  # temiz veri seti

print(f'İade olarak ayrılan : {len(df_returns):,} satır')
print(f'Kalan               : {len(df_clean):,} satır')

İade olarak ayrılan : 0 satır
Kalan               : 387,841 satır


In [3]:
# ADIM 3 — Sıfır ve negatif UnitPrice
# Fiyatı olmayan ürünler promosyon veya veri hatası olabilir

mask_price = df_clean['UnitPrice'] <= 0
print(f'Çıkarılan : {mask_price.sum():,} satır')

df_clean = df_clean[~mask_price].copy()
print(f'Kalan     : {len(df_clean):,} satır')

Çıkarılan : 40 satır
Kalan     : 406,789 satır


In [7]:
# ADIM 4 — 'C' ile başlayan iptal faturaları
# InvoiceNo 'C' ile başlıyorsa o işlem iptal/iade faturasıdır

mask_cancel  = df_clean['InvoiceNo'].astype(str).str.startswith('C')
df_cancelled = df_clean[mask_cancel].copy()
df_clean     = df_clean[~mask_cancel].copy()

print(f'İptal faturası : {len(df_cancelled):,} satır')
print(f'Kalan          : {len(df_clean):,} satır')

İptal faturası : 8,806 satır
Kalan          : 387,841 satır


In [4]:
# ADIM 5 — Duplicate kayıtlar
# Aynı fatura + ürün + müşteri kombinasyonu birden fazla kez geçiyorsa
# yalnızca bir tanesi tutulur

once = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['InvoiceNo', 'StockCode', 'CustomerID'])

print(f'Çıkarılan : {once - len(df_clean):,} satır')
print(f'Kalan     : {len(df_clean):,} satır')

Çıkarılan : 10,142 satır
Kalan     : 396,647 satır


In [5]:
# ADIM 6 — Aykırı değerler (Winsorization)
# Toptan satıcıların çok büyük siparişleri aykırı değer yaratır
# Kırpmak yerine %99 eşiğiyle sınırlandırıyoruz (winsorization)

q99_qty   = df_clean['Quantity'].quantile(0.99)
q99_price = df_clean['UnitPrice'].quantile(0.99)

print(f'Quantity  %99 eşiği : {q99_qty:.0f}')
print(f'UnitPrice %99 eşiği : {q99_price:.2f} £')

df_clean['Quantity']  = df_clean['Quantity'].clip(upper=q99_qty)
df_clean['UnitPrice'] = df_clean['UnitPrice'].clip(upper=q99_price)

print('Winsorization uygulandı ✅')

Quantity  %99 eşiği : 120
UnitPrice %99 eşiği : 15.00 £
Winsorization uygulandı ✅


In [10]:
# ÖZET ve kaydet

print('=' * 45)
print('TEMİZLEME ÖZETİ')
print('=' * 45)
print(f'Başlangıç satırı      : {541909:>10,}')
print(f'Eksik CustomerID (-)  : {135080:>10,}')
print(f'Negatif Quantity (-)  : {  8905:>10,}')
print(f'Sıfır UnitPrice (-)   : {    40:>10,}')
print(f'İptal faturaları (-)  : {len(df_cancelled):>10,}')
print(f'Duplicate (-)         : {10043:>10,}')
print('-' * 45)
print(f'Temiz veri seti       : {len(df_clean):>10,}')
print(f'Benzersiz müşteri     : {df_clean["CustomerID"].nunique():>10,}')
print(f'Benzersiz fatura      : {df_clean["InvoiceNo"].nunique():>10,}')

# Dosyalara yaz
df_clean.to_csv('data_clean.csv', index=False)
df_returns.to_csv('data_returns.csv', index=False)

print('\n✅ data_clean.csv   kaydedildi')
print('✅ data_returns.csv kaydedildi')

TEMİZLEME ÖZETİ
Başlangıç satırı      :    541,909
Eksik CustomerID (-)  :    135,080
Negatif Quantity (-)  :      8,905
Sıfır UnitPrice (-)   :         40
İptal faturaları (-)  :      8,806
Duplicate (-)         :     10,043
---------------------------------------------
Temiz veri seti       :    387,841
Benzersiz müşteri     :      4,338
Benzersiz fatura      :     18,532

✅ data_clean.csv   kaydedildi
✅ data_returns.csv kaydedildi
